# Лабораторная работа № 2. Линейная регрессия: МНК, Ridge, Lasso

**Курс:** Классическое машинное обучение, 4 курс прикладной математики

## Цель работы

Реализовать линейные модели регрессии с нуля и с использованием готовых библиотек, сравнить различные виды регуляризации.

**Используемые инструменты:** `numpy`, `scipy.linalg.svd`, `sklearn.linear_model`, `sklearn.model_selection`.

### Регламент сдачи

Работа сдаётся в виде этого же ноутбука, дополненного вашим кодом. Обязательно:

1. Читаемый код с комментариями.
2. Визуализации (графики, таблицы).
3. **Текстовый вывод после каждого задания** — не только код, но и объяснение результата.
4. Финальный вывод по работе.

**Критерии оценки:** корректность реализации — 30 %, качество визуализаций и анализа — 20 %,
обоснованность выводов — 20 %, сравнение с эталонными реализациями — 15 %,
оригинальность и дополнительная работа — 15 %.

> Ячейки, помеченные `# TODO`, нужно заполнить самостоятельно.
> Ячейки с готовым кодом можно просто выполнить — они подготавливают данные и графики.

## Подготовка окружения

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})
sns.set_palette("viridis")

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

data = load_diabetes()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print("train:", X_train.shape, " test:", X_test.shape)

## Задание 1. Класс `LinearRegressionSVD`

Решите задачу МНК через сингулярное разложение $X = U \Sigma V^T$. Псевдообратная матрица:

$$X^{+} = V \Sigma^{+} U^T, \qquad \hat{w} = X^{+} y,$$

где $\Sigma^{+}$ получается транспонированием $\Sigma$ и обращением ненулевых сингулярных чисел.

**Почему не нормальное уравнение?** Число обусловленности $X^T X$ равно квадрату числа
обусловленности $X$ — при почти линейно зависимых признаках прямое обращение теряет точность.

In [ ]:
class LinearRegressionSVD:
    """МНК через сингулярное разложение."""

    def __init__(self, fit_intercept=True, rcond=1e-10):
        self.fit_intercept = fit_intercept
        self.rcond = rcond          # порог отсечения малых сингулярных чисел
        self.w = None
        self.intercept_ = 0.0

    def _add_intercept(self, X):
        # TODO: если fit_intercept — добавьте столбец из единиц
        raise NotImplementedError

    def fit(self, X, y):
        # TODO:
        #  1. при необходимости добавьте столбец единиц
        #  2. U, s, Vt = np.linalg.svd(X, full_matrices=False)
        #  3. обнулите вклад сингулярных чисел меньше rcond * s.max()
        #  4. w = Vt.T @ np.diag(s_inv) @ U.T @ y
        raise NotImplementedError

    def predict(self, X):
        # TODO
        raise NotImplementedError


# Проверка: сравните с np.linalg.lstsq и с sklearn.linear_model.LinearRegression

## Задание 2. Класс `RidgeRegression`

Аналитическое решение гребневой регрессии:

$$\hat{w} = (X^T X + \lambda I)^{-1} X^T y.$$

Свободный член штрафовать **не нужно** — обнулите соответствующий элемент на диагонали $I$.

In [ ]:
class RidgeRegression:
    """Гребневая регрессия по аналитической формуле."""

    def __init__(self, lam=1.0, fit_intercept=True):
        self.lam = lam
        self.fit_intercept = fit_intercept
        self.w = None

    def fit(self, X, y):
        # TODO: реализуйте формулу выше.
        #       Используйте np.linalg.solve(A, b), а не np.linalg.inv(A) @ b — точнее и быстрее.
        raise NotImplementedError

    def predict(self, X):
        # TODO
        raise NotImplementedError


# TODO: проверьте, что при lam -> 0 решение совпадает с МНК,
#       а при lam -> inf все веса стремятся к нулю

## Задание 3. Lasso

Для Lasso используйте `sklearn.linear_model.Lasso` (собственная реализация — в дополнительном задании).

In [ ]:
from sklearn.linear_model import Lasso, Ridge, LinearRegression

# TODO: обучите Lasso с несколькими значениями alpha и посмотрите,
#       сколько коэффициентов обратилось строго в нуль

## Задание 4. Подбор $\lambda$ и графики коэффициентов

1. Обучите все три модели на `diabetes`.
2. Подберите оптимальное $\lambda$ по 5-fold кросс-валидации.
3. Постройте **траектории коэффициентов**: по оси $x$ — $\lambda$ в логарифмическом масштабе,
   по оси $y$ — значения весов.

In [ ]:
lambdas = np.logspace(-3, 3, 30)

# TODO: для каждой lambda обучите Ridge и Lasso, сохраните веса,
#       постройте два графика траекторий рядом (plt.subplots(1, 2))
#       и отметьте вертикальной линией оптимальное значение по CV

In [ ]:
# TODO: подбор lambda по 5-fold кросс-валидации (KFold + cross_val_score,
#       scoring="neg_mean_squared_error"). Постройте график MSE(lambda).

## Задание 5. Сравнение MSE на тесте

Соберите итоговую таблицу: модель | лучшее λ | MSE train | MSE test | число ненулевых весов.

In [ ]:
# TODO: постройте pandas.DataFrame со сравнением всех моделей

## Задание 6. Выводы о разреженности и стабильности

**Вывод:** *ответьте по пунктам:*

1. Почему Lasso зануляет коэффициенты, а Ridge — нет? (подсказка: форма шара $\ell_1$ против $\ell_2$)
2. Какая модель устойчивее при сильно скоррелированных признаках и почему?
3. Что происходит с траекториями коэффициентов при росте λ?

## Дополнительное задание. Lasso координатным спуском

Реализуйте координатный спуск: на каждом шаге обновляем один вес при фиксированных остальных

$$w_j^{\text{new}} = \operatorname{soft}\!\left(\frac{\sum_i x_{ij}\bigl(y_i - \sum_{k \neq j} x_{ik} w_k\bigr)}{n},\ \lambda\right),$$

где мягкое пороговое преобразование $\operatorname{soft}(z, \lambda) = \operatorname{sign}(z)\cdot\max(|z| - \lambda,\, 0)$.

In [ ]:
def soft_threshold(z, lam):
    # TODO: np.sign(z) * np.maximum(np.abs(z) - lam, 0)
    raise NotImplementedError


class LassoCoordinateDescent:
    def __init__(self, lam=1.0, n_iter=100, tol=1e-6):
        self.lam, self.n_iter, self.tol = lam, n_iter, tol

    def fit(self, X, y):
        # TODO: циклом по признакам обновляйте веса до сходимости.
        #       Признаки предварительно стандартизуйте.
        raise NotImplementedError


# TODO: сравните полученные веса со sklearn.linear_model.Lasso (должны почти совпасть)

## Финальный вывод

*Напишите здесь связный вывод по работе (5–10 предложений):*

- какие методы вы применили и почему;
- какие результаты получили в числах;
- где реализация «с нуля» разошлась с эталоном из `sklearn` и в чём причина;
- что бы вы улучшили, будь у вас больше времени.